In [ ]:
import numpy as np
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
!pip3 install -U ydata-profiling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
from scipy import stats

In [ ]:
ruta = '/content'

### Lectura desde archivos CSV en dataframe

In [ ]:
df = pd.read_csv('%s/developer_stress.csv' % ruta)
df.head()

In [ ]:
# Número de Filas
df.describe()

### Reporte de Perfilamiento (Pandas Profiling)

In [ ]:
# Reporte
profile = ProfileReport(df, title='Developer Stress - Reporte de Perfilamiento')

In [ ]:
profile

In [ ]:
profile.to_file(ruta + '/developer_stress_profile.html')

### Análisis Descriptivo

In [ ]:
# Tipos de datos y valores nulos
print('Tipos de datos:')
print(df.dtypes)
print('\nValores nulos por columna:')
print(df.isnull().sum())

In [ ]:
# Separar variables numéricas y categóricas
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
print('Variables numéricas:', num_cols)
print('Variables categóricas:', cat_cols)

In [ ]:
# Estadísticas descriptivas extendidas
df[num_cols].describe().T.assign(
    skewness=df[num_cols].skew(),
    kurtosis=df[num_cols].kurt()
)

In [ ]:
# Histogramas de variables numéricas
df[num_cols].hist(figsize=(14, 10), bins=20, edgecolor='black', color='steelblue')
plt.suptitle('Distribución de Variables Numéricas', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots de variables numéricas
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(14, 10))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='steelblue', color='navy'),
                    medianprops=dict(color='red'))
    axes[i].set_title(col)
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Boxplots - Variables Numéricas', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Distribución de variables categóricas
fig, axes = plt.subplots(1, len(cat_cols), figsize=(14, 4))
if len(cat_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, cat_cols):
    df[col].value_counts().plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title(f'Distribución: {col}')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlación
plt.figure(figsize=(10, 7))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Matriz de Correlación', fontsize=14)
plt.tight_layout()
plt.show()

### Detección de Outliers

In [ ]:
# Método IQR (Rango Intercuartílico)
print('=== Método IQR ===')
outliers_iqr = {}
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    outliers_iqr[col] = len(outliers)
    print(f'{col}: {len(outliers)} outliers  |  Límite inferior: {lower:.2f}  |  Límite superior: {upper:.2f}')

pd.Series(outliers_iqr, name='Outliers (IQR)').to_frame()

In [ ]:
# Método Z-Score
print('=== Método Z-Score (umbral |z| > 3) ===')
outliers_z = {}
for col in num_cols:
    z_scores = np.abs(stats.zscore(df[col].dropna()))
    n_outliers = (z_scores > 3).sum()
    outliers_z[col] = n_outliers
    print(f'{col}: {n_outliers} outliers')

pd.Series(outliers_z, name='Outliers (Z-Score)').to_frame()

In [ ]:
# Comparación de métodos de detección
comparison = pd.DataFrame({
    'IQR': outliers_iqr,
    'Z-Score': outliers_z
})
comparison['% IQR'] = (comparison['IQR'] / len(df) * 100).round(2)
comparison['% Z-Score'] = (comparison['Z-Score'] / len(df) * 100).round(2)
comparison

In [ ]:
# Visualización de outliers con IQR
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(14, 10))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    is_outlier = (df[col] < lower) | (df[col] > upper)
    axes[i].scatter(range(len(df)), df[col], c=is_outlier.map({True: 'red', False: 'steelblue'}),
                    alpha=0.6, s=15)
    axes[i].axhline(lower, color='orange', linestyle='--', linewidth=1, label='Límites IQR')
    axes[i].axhline(upper, color='orange', linestyle='--', linewidth=1)
    axes[i].set_title(f'{col} (outliers en rojo)')
    axes[i].set_xlabel('')
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Detección de Outliers - Método IQR', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Visualización de outliers con Z-Score
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(14, 10))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    z = np.abs(stats.zscore(df[col]))
    is_outlier = z > 3
    axes[i].scatter(range(len(df)), df[col], c=pd.Series(is_outlier).map({True: 'red', False: 'steelblue'}),
                    alpha=0.6, s=15)
    axes[i].set_title(f'{col} (|z|>3 en rojo)')
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Detección de Outliers - Método Z-Score', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Resumen visual de outliers por variable
comparison[['IQR', 'Z-Score']].plot(kind='bar', figsize=(12, 5), color=['steelblue', 'tomato'],
                                      edgecolor='black')
plt.title('Cantidad de Outliers por Variable y Método', fontsize=13)
plt.xlabel('Variable')
plt.ylabel('N° de Outliers')
plt.xticks(rotation=30)
plt.legend(title='Método')
plt.tight_layout()
plt.show()